# Diffusion Model (DDPM) - TensorFlow / Keras

**Goal:** Denoise samples with a minimal diffusion-style objective.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** Forward noising and learned denoising create a generative process.
- **Where it is used:** image/audio generation, denoising, and conditional generation.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Diffusion Model: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['clean', 'noise', 'denoise']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = np.sqrt(np.linspace(1,.25,160))*np.sin(x)
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.10,.35,.65,.90])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['t0', 't25', 't50', 't99'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow: {tf.__version__}")
from sklearn.datasets import load_digits

X = (load_digits().data / 16.0).astype("float32")
dataset = tf.data.Dataset.from_tensor_slices(X).shuffle(1024).batch(64)
timesteps = 100
betas = tf.linspace(1e-4, 0.02, timesteps)
alphas = 1.0 - betas
alpha_bars = tf.math.cumprod(alphas)


In [ ]:
clean_input = keras.Input(shape=(64,))
time_input = keras.Input(shape=(), dtype="int32")
t = layers.Embedding(timesteps, 32)(time_input)
x = layers.Concatenate()([clean_input, t])
x = layers.Dense(128, activation="swish")(x)
x = layers.Dense(128, activation="swish")(x)
predicted_noise = layers.Dense(64)(x)
model = keras.Model([clean_input, time_input], predicted_noise)
optimizer = keras.optimizers.AdamW(1e-3)


In [ ]:
for epoch in range(20):
    losses = []
    for clean_x in dataset:
        batch_size = tf.shape(clean_x)[0]
        t = tf.random.uniform((batch_size,), minval=0, maxval=timesteps, dtype=tf.int32)
        noise = tf.random.normal(tf.shape(clean_x))
        alpha_bar = tf.gather(alpha_bars, t)[:, None]
        noisy_x = tf.sqrt(alpha_bar) * clean_x + tf.sqrt(1 - alpha_bar) * noise
        with tf.GradientTape() as tape:
            loss = tf.reduce_mean(tf.square(model([noisy_x, t], training=True) - noise))
        optimizer.apply_gradients(zip(tape.gradient(loss, model.trainable_variables), model.trainable_variables))
        losses.append(float(loss))
    print(f"epoch={epoch+1:02d} denoise_mse={np.mean(losses):.4f}")
